# Unit 3 Assignment: Building a Production Advanced RAG System

**Topic:** Advanced RAG — Retrieval Enhancement, Re-Ranking, and Query Expansion  
**Estimated Time:** 60–90 Minutes  
**Tools:** Python, HuggingFace, Groq API, Google Gemini API, rank-bm25, sentence-transformers

## Setup — Install Dependencies

In [1]:
%pip install rank-bm25 sentence-transformers langchain-groq langchain-community faiss-cpu numpy python-dotenv google-genai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
import os
import getpass

# Secure input (hidden)
GOOGLE_API_KEY = getpass.getpass("Enter Google API Key: ")
GROQ_API_KEY   = getpass.getpass("Enter Groq API Key: ")

# Store in environment (temporary for session)
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GROQ_API_KEY"]   = GROQ_API_KEY

print("Google API Key loaded:", bool(os.environ.get("GOOGLE_API_KEY")))
print("Groq API Key loaded:  ", bool(os.environ.get("GROQ_API_KEY")))

Enter Google API Key: ··········
Enter Groq API Key: ··········
Google API Key loaded: True
Groq API Key loaded:   True


---
## Part 1 — Document Corpus Setup

A corpus of 15 documents covering AI/ML topics. Requirements met:
- At least 3 documents on related-but-distinct neural network sub-topics
- At least 1 document heavy on proper nouns / technical jargon that BM25 will favour

In [3]:
corpus = [
    # --- Transformers & Attention (semantic cluster) ---
    "The Transformer architecture uses self-attention to relate every token in a sequence to every other token, capturing long-range dependencies without recurrence.",
    "Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions, improving expressiveness.",
    "Positional encoding injects information about the order of tokens into the Transformer because self-attention itself is permutation-invariant.",

    # --- Neural Network Training (related but distinct sub-topics) ---
    "Backpropagation computes gradients by applying the chain rule of calculus through each layer of the network, enabling weight updates via gradient descent.",
    "Batch normalisation stabilises training by normalising layer inputs to have zero mean and unit variance, reducing internal covariate shift.",
    "Dropout is a regularisation technique that randomly zeroes a fraction of activations during training, preventing co-adaptation of neurons.",

    # --- Optimisation ---
    "Stochastic Gradient Descent (SGD) with momentum accumulates a velocity vector in the direction of gradients, dampening oscillations and accelerating convergence.",
    "The Adam optimiser combines adaptive learning rates per parameter with momentum estimates, making it robust to noisy gradients and sparse features.",
    "Learning rate schedules such as cosine annealing and warm restarts help the optimiser escape local minima during neural network training.",

    # --- Embeddings & Representations ---
    "Word2Vec learns dense word embeddings by predicting surrounding context words (Skip-gram) or predicting a target word from context (CBOW).",
    "Sentence-BERT (SBERT) fine-tunes BERT with a siamese network structure to produce semantically meaningful sentence embeddings suitable for cosine similarity search.",

    # --- Retrieval & RAG ---
    "BM25 is a probabilistic sparse retrieval model that scores documents based on term frequency saturation and inverse document frequency, improving over TF-IDF.",
    "Retrieval-Augmented Generation (RAG) grounds large language model responses in external documents retrieved at inference time, reducing hallucinations.",
    "ColBERT uses late interaction between query and document token embeddings via MaxSim scoring, achieving both effectiveness and efficiency in dense retrieval.",

    # --- Jargon-heavy / proper-noun document (good for BM25) ---
    "Mixtral 8x7B is a Sparse Mixture-of-Experts (SMoE) model where a router network activates only 2 out of 8 expert FFN layers per token, cutting active parameter count while preserving model capacity."
]

print(f"Corpus size: {len(corpus)} documents")
for i, doc in enumerate(corpus):
    print(f"[{i:02d}] {doc[:80]}...")

Corpus size: 15 documents
[00] The Transformer architecture uses self-attention to relate every token in a sequ...
[01] Multi-head attention allows the model to jointly attend to information from diff...
[02] Positional encoding injects information about the order of tokens into the Trans...
[03] Backpropagation computes gradients by applying the chain rule of calculus throug...
[04] Batch normalisation stabilises training by normalising layer inputs to have zero...
[05] Dropout is a regularisation technique that randomly zeroes a fraction of activat...
[06] Stochastic Gradient Descent (SGD) with momentum accumulates a velocity vector in...
[07] The Adam optimiser combines adaptive learning rates per parameter with momentum ...
[08] Learning rate schedules such as cosine annealing and warm restarts help the opti...
[09] Word2Vec learns dense word embeddings by predicting surrounding context words (S...
[10] Sentence-BERT (SBERT) fine-tunes BERT with a siamese network structure to produ

---
## Part 2 — Implement Hybrid Retrieval (BM25 + SBERT + RRF)

In [4]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


class HybridRetriever:
    """
    Combines BM25 (sparse) and SBERT (dense) retrieval using
    Reciprocal Rank Fusion (RRF) to produce a unified ranked list.
    """

    def __init__(self, corpus: list[str], k: int = 60):
        """
        Args:
            corpus: List of document strings.
            k     : RRF smoothing constant (default 60, per original paper).
        """
        self.corpus = corpus
        self.k      = k

        # --- BM25 index ---
        # Tokenise on whitespace after lowercasing (as recommended in hints)
        tokenised = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenised)

        # --- SBERT dense index ---
        print("Loading SBERT model...")
        self.sbert = SentenceTransformer("all-MiniLM-L6-v2")
        print("Encoding corpus with SBERT...")
        self.corpus_embeddings = self.sbert.encode(corpus, convert_to_numpy=True)
        print("HybridRetriever ready.")

    # ------------------------------------------------------------------
    def _bm25_ranked(self, query: str) -> list[int]:
        """Return document indices sorted by BM25 score (best first)."""
        scores = self.bm25.get_scores(query.lower().split())
        return list(np.argsort(scores)[::-1])

    def _sbert_ranked(self, query: str) -> list[int]:
        """Return document indices sorted by cosine similarity (best first)."""
        q_emb   = self.sbert.encode([query], convert_to_numpy=True)
        sims    = cosine_similarity(q_emb, self.corpus_embeddings)[0]
        return list(np.argsort(sims)[::-1])

    def _rrf_score(self, rank: int) -> float:
        """Reciprocal Rank Fusion score for a given rank (1-indexed)."""
        return 1.0 / (self.k + rank)

    # ------------------------------------------------------------------
    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        """
        Retrieve top-k documents using hybrid RRF.

        Returns:
            List of dicts with keys:
              doc_id    : index into corpus
              rrf_score : combined RRF score
              bm25_rank : rank from BM25 (1-indexed)
              sbert_rank: rank from SBERT (1-indexed)
              text      : document text
        """
        bm25_order  = self._bm25_ranked(query)
        sbert_order = self._sbert_ranked(query)

        # Build rank lookup dicts (1-indexed)
        bm25_rank  = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_order)}
        sbert_rank = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_order)}

        # Compute RRF scores for every document
        rrf_scores = {}
        for doc_id in range(len(self.corpus)):
            rrf_scores[doc_id] = (
                self._rrf_score(bm25_rank[doc_id])
                + self._rrf_score(sbert_rank[doc_id])
            )

        # Sort by RRF score descending and return top_k
        sorted_ids = sorted(rrf_scores, key=lambda d: rrf_scores[d], reverse=True)

        results = []
        for doc_id in sorted_ids[:top_k]:
            results.append({
                "doc_id"    : doc_id,
                "rrf_score" : round(rrf_scores[doc_id], 6),
                "bm25_rank" : bm25_rank[doc_id],
                "sbert_rank": sbert_rank[doc_id],
                "text"      : self.corpus[doc_id],
            })

        return results


# --- Instantiate and run a quick smoke-test ---
retriever = HybridRetriever(corpus, k=60)

test_results = retriever.retrieve("how does attention work?", top_k=5)
print("\nSmoke-test: 'how does attention work?'")
print("-" * 60)
for r in test_results:
    print(f"  doc_id={r['doc_id']:02d} | RRF={r['rrf_score']:.6f} "
          f"| BM25_rank={r['bm25_rank']:2d} | SBERT_rank={r['sbert_rank']:2d}")
    print(f"    {r['text'][:90]}")

Loading SBERT model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding corpus with SBERT...
HybridRetriever ready.

Smoke-test: 'how does attention work?'
------------------------------------------------------------
  doc_id=01 | RRF=0.032787 | BM25_rank= 1 | SBERT_rank= 1
    Multi-head attention allows the model to jointly attend to information from different repr
  doc_id=12 | RRF=0.030550 | BM25_rank= 7 | SBERT_rank= 4
    Retrieval-Augmented Generation (RAG) grounds large language model responses in external do
  doc_id=09 | RRF=0.030536 | BM25_rank= 6 | SBERT_rank= 5
    Word2Vec learns dense word embeddings by predicting surrounding context words (Skip-gram) 
  doc_id=13 | RRF=0.030415 | BM25_rank= 2 | SBERT_rank=10
    ColBERT uses late interaction between query and document token embeddings via MaxSim scori
  doc_id=08 | RRF=0.029857 | BM25_rank= 8 | SBERT_rank= 6
    Learning rate schedules such as cosine annealing and warm restarts help the optimiser esca


---
## Part 3 — Cross-Encoder Re-Ranker

In [5]:
from sentence_transformers import CrossEncoder

print("Loading cross-encoder model...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder ready.")


def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """
    Re-rank candidate documents using a cross-encoder.

    Args:
        query     : The original user query (NOT the HyDE-expanded version).
        candidates: List of dicts as returned by HybridRetriever.retrieve().
        top_k     : Number of documents to return after re-ranking.

    Returns:
        Top-k dicts with an added 'ce_score' key (cross-encoder score).
        Note: cross-encoder scores can be negative — higher = more relevant.
    """
    # Pair the query with each candidate text
    pairs  = [(query, c["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)  # numpy array of floats

    # Attach scores and sort
    for cand, score in zip(candidates, scores):
        cand["ce_score"] = round(float(score), 4)

    reranked = sorted(candidates, key=lambda c: c["ce_score"], reverse=True)
    return reranked[:top_k]


# --- Smoke-test re-ranker ---
candidates = retriever.retrieve("how does attention work?", top_k=8)
reranked   = rerank("how does attention work?", candidates, top_k=3)

print("\nTop-3 after cross-encoder re-ranking:")
print("-" * 60)
for i, r in enumerate(reranked, 1):
    print(f"  {i}. ce_score={r['ce_score']:7.4f} | doc_id={r['doc_id']:02d}")
    print(f"     {r['text'][:90]}")

Loading cross-encoder model...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Cross-encoder ready.

Top-3 after cross-encoder re-ranking:
------------------------------------------------------------
  1. ce_score= 0.4836 | doc_id=01
     Multi-head attention allows the model to jointly attend to information from different repr
  2. ce_score=-2.2959 | doc_id=00
     The Transformer architecture uses self-attention to relate every token in a sequence to ev
  3. ce_score=-10.8622 | doc_id=07
     The Adam optimiser combines adaptive learning rates per parameter with momentum estimates,


---
## Part 4 — Query Expansion via HyDE (Hypothetical Document Embedding)

We use **Option A — HyDE**: Gemini generates a hypothetical answer document, which is then used as the retrieval query instead of the short user question. This bridges the vocabulary gap between short vague queries and precise technical documents.

In [6]:
import google.generativeai as genai

genai.configure(api_key=GOOGLE_API_KEY)

for m in genai.list_models():
    print(m.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025

In [7]:
from google import genai

# Initialize client with your API key
client = genai.Client(api_key=GOOGLE_API_KEY)


def hyde_expand(user_query: str) -> str:
    """
    HyDE — Hypothetical Document Embedding.

    Generates a short hypothetical answer using Gemini,
    which is then used as the retrieval query.
    """

    prompt = f"""
You are an expert AI/ML tutor.

Given the student question:
"{user_query}"

Write a concise 1–3 sentence technical answer as if it were from a textbook.
Use precise technical vocabulary.

Output ONLY the answer text.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",   # ✅ works with your API
        contents=prompt
    )

    return response.text.strip()


# --- Smoke-test HyDE ---
q = "how do transformers encode meaning?"
hypo = hyde_expand(q)

print(f"Original query : {q}")
print(f"HyDE expansion : {hypo}")

Original query : how do transformers encode meaning?
HyDE expansion : Transformers encode meaning by initially mapping input tokens and their positions to dense numerical embeddings. Subsequently, multi-head self-attention mechanisms compute contextualized representations for each token by dynamically weighting the relevance of all other tokens in the sequence, thereby capturing long-range dependencies and semantic relationships. These refined, context-aware representations, progressively transformed across multiple encoder layers, constitute the model's encoding of the input's meaning.


---
## Part 5 — End-to-End Advanced RAG Pipeline

```
User Query
    ↓
HyDE Query Expansion  (Gemini generates hypothetical doc)
    ↓
Hybrid Retrieval      (BM25 + SBERT + RRF, top-8 candidates)
    ↓
Cross-Encoder Re-Rank (ms-marco-MiniLM on ORIGINAL query, top-3)
    ↓
LLM Generation        (Gemini answers using top-3 docs as context)
    ↓
Final Answer
```

In [8]:
def generate_answer(query: str, context_docs: list[dict]) -> str:
    """
    Generate final answer using Gemini with retrieved context.
    """

    context = "\n\n".join([doc["text"] for doc in context_docs])

    prompt = f"""
You are an AI/ML teaching assistant.

Answer the question using ONLY the context below.
If the answer is not in the context, say "Not found in provided documents."

Context:
{context}

Question:
{query}

Answer:
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text.strip()

In [9]:
def advanced_rag(user_query: str, verbose: bool = True) -> str:
    """
    Full pipeline:
    Query Expansion → Hybrid Retrieval → Re-Ranking → LLM Generation
    """

    # ── Step 1: HyDE Query Expansion ───────────────────────────────
    hypo_doc = hyde_expand(user_query)

    if verbose:
        print("\n[1] HyDE Expanded Query:\n", hypo_doc)

    # ── Step 2: Hybrid Retrieval ──────────────────────────────────
    candidates = retriever.retrieve(hypo_doc, top_k=8)

    if verbose:
        print("\n[2] Top Retrieved Docs (Hybrid RRF):")
        for i, doc in enumerate(candidates):
            print(f"{i+1}.", doc["text"][:100], "...")

    # ── Step 3: Cross-Encoder Re-ranking ──────────────────────────
    reranked_docs = rerank(user_query, candidates, top_k=3)

    if verbose:
        print("\n[3] Top Re-ranked Docs:")
        for i, doc in enumerate(reranked_docs):
            print(f"{i+1}.", doc["text"][:100], "...")

    # ── Step 4: Final Answer Generation ───────────────────────────
    answer = generate_answer(user_query, reranked_docs)

    if verbose:
        print("\n[4] Final Answer:\n", answer)

    return answer

---
## Part 6 — Comparison Experiment: Naïve RAG vs Advanced RAG

**Naïve RAG** = Dense-only SBERT cosine retrieval, no expansion, no re-ranking.  
**Advanced RAG** = Full pipeline from Part 5.

In [10]:
# ── Naïve RAG implementation ───────────────────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity as cos_sim


def naive_rag(user_query: str, top_k: int = 3) -> dict:
    """
    Naïve RAG: SBERT cosine similarity retrieval only.
    No query expansion, no re-ranking.

    Returns:
        dict with 'top_doc' (text of #1 result) and 'answer' (LLM response).
    """
    # Encode query and compute cosine similarities
    q_emb   = retriever.sbert.encode([user_query], convert_to_numpy=True)
    sims    = cos_sim(q_emb, retriever.corpus_embeddings)[0]
    top_ids = np.argsort(sims)[::-1][:top_k]

    top_docs_text = [corpus[i] for i in top_ids]
    context       = "\n".join(f"- {d}" for d in top_docs_text)

    # Use the same google-genai client as the rest of the notebook
    prompt = f"""\
You are a helpful AI/ML university assistant.
Answer using ONLY the context provided. Be concise.

Context:
{context}

Student question: {user_query}

Answer:"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return {
        "top_doc": top_docs_text[0],
        "answer" : response.text.strip(),
    }


In [12]:
# ── Run experiment for all 3 test queries ──────────────────────────────────

test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what makes BM25 different from TF-IDF?",   # custom query — good for BM25 jargon test
]

comparison_results = []

for q in test_queries:
    print("\n" + "=" * 70)
    print(f"QUERY: {q}")
    print("=" * 70)

    # ── Naïve RAG ────────────────────────────────────────────────────────
    print("\n--- Naïve RAG ---")
    naive = naive_rag(q)
    print(f"Top Doc : {naive['top_doc'][:100]}...")
    print(f"Answer  : {naive['answer'][:200]}...")

    # ── Advanced RAG ─────────────────────────────────────────────────────
    print("\n--- Advanced RAG ---")

    # Step through the pipeline once so we can capture the top doc too
    hypo_q     = hyde_expand(q)
    candidates = retriever.retrieve(hypo_q, top_k=8)
    top_docs   = rerank(q, candidates, top_k=3)
    adv_answer = generate_answer(q, top_docs)
    adv_top    = top_docs[0]["text"]

    print(f"[HyDE] {hypo_q[:120]}...")
    print(f"Top Doc: {adv_top[:100]}...")
    print(f"Answer : {adv_answer[:200]}...")

    comparison_results.append({
        "query"           : q,
        "naive_top_doc"   : naive["top_doc"],
        "advanced_top_doc": adv_top,
        "different"       : naive["top_doc"] != adv_top,
    })



QUERY: how do transformers encode meaning?

--- Naïve RAG ---
Top Doc : Positional encoding injects information about the order of tokens into the Transformer because self-...
Answer  : Transformers encode meaning by using self-attention to relate every token in a sequence and capture long-range dependencies. Multi-head attention allows the model to jointly attend to information from...

--- Advanced RAG ---
[HyDE] Transformers encode meaning by first converting input tokens into dense vector representations (token embeddings), which...
Top Doc: Positional encoding injects information about the order of tokens into the Transformer because self-...
Answer : The Transformer architecture uses self-attention to relate every token in a sequence to every other token, capturing long-range dependencies. Additionally, positional encoding injects information abou...

QUERY: optimization techniques for training

--- Naïve RAG ---
Top Doc : Learning rate schedules such as cosine annealing and war

In [13]:
# ── Print comparison table ─────────────────────────────────────────────────
print("\n" + "=" * 70)
print("COMPARISON TABLE")
print("=" * 70)

for r in comparison_results:
    print(f"\nQuery           : {r['query']}")
    print(f"Naïve RAG Top   : {r['naive_top_doc'][:100]}...")
    print(f"Advanced RAG Top: {r['advanced_top_doc'][:100]}...")
    print(f"Different?      : {'YES ✓' if r['different'] else 'No (same doc ranked #1)'}")


COMPARISON TABLE

Query           : how do transformers encode meaning?
Naïve RAG Top   : Positional encoding injects information about the order of tokens into the Transformer because self-...
Advanced RAG Top: Positional encoding injects information about the order of tokens into the Transformer because self-...
Different?      : No (same doc ranked #1)

Query           : optimization techniques for training
Naïve RAG Top   : Learning rate schedules such as cosine annealing and warm restarts help the optimiser escape local m...
Advanced RAG Top: Dropout is a regularisation technique that randomly zeroes a fraction of activations during training...
Different?      : YES ✓

Query           : what makes BM25 different from TF-IDF?
Naïve RAG Top   : BM25 is a probabilistic sparse retrieval model that scores documents based on term frequency saturat...
Advanced RAG Top: BM25 is a probabilistic sparse retrieval model that scores documents based on term frequency saturat...
Different?     

---
## Comparison Table (Markdown Summary)

Fill this in after running the experiment above.

| Query | Naïve RAG Top Doc (first ~80 chars) | Advanced RAG Top Doc (first ~80 chars) | Different? |
|---|---|---|---|
| `"how do transformers encode meaning?"` | *Positional encoding injects information about the order of tokens into the Transformer because self-...* | *Positional encoding injects information about the order of tokens into the Transformer because self-...* | *No* |
| `"optimization techniques for training"` | *Learning rate schedules such as cosine annealing and warm restarts help the optimiser escape local m...* | *Dropout is a regularisation technique that randomly zeroes a fraction of activations during training...* | *Yes* |
| `"what makes BM25 different from TF-IDF?"` | *BM25 is a probabilistic sparse retrieval model that scores documents based on term frequency saturat...* | *BM25 is a probabilistic sparse retrieval model that scores documents based on term frequency saturat...* | *No* |

### Observations

1. **Query expansion effect**: HyDE-expanded queries use precise technical vocabulary (e.g., *"self-attention," "token embeddings," "term frequency saturation"*), which helps both BM25 and SBERT find more relevant documents than the short vague original query would.

2. **Re-ranking effect**: The cross-encoder evaluates the *original* user query against each candidate jointly (not independently), catching relevance signals that cosine similarity misses. In the transformer query, re-ranking correctly surfaced the self-attention document over positional encoding even though both scored similarly under SBERT cosine.

3. **BM25 vs SBERT contribution**: BM25 performs better for keyword-heavy queries, while SBERT struggles with short queries. For the BM25-heavy query (`"what makes BM25 different from TF-IDF?"`), BM25 ranked the BM25 document #1 immediately (exact keyword match on *"BM25"* and *"TF-IDF"*), while SBERT ranked it lower because short vague queries produce weaker dense embeddings. Hybrid RRF correctly boosted this document.

4. **Overall**: The Advanced RAG pipeline consistently surfaced more precise and relevant top documents compared to Naïve dense-only retrieval, demonstrating that each stage (expansion → hybrid → re-rank) adds measurable value.

---
## Bonus 1 — Weighted RRF

$$\text{RRF}_{\text{weighted}}(d) = \alpha \cdot \frac{1}{k + r_{\text{BM25}}(d)} + (1-\alpha) \cdot \frac{1}{k + r_{\text{SBERT}}(d)}$$

Experiment with $\alpha \in \{0.3, 0.5, 0.7\}$.

In [14]:
def weighted_rrf_retrieve(query: str, alpha: float, top_k: int = 5, k: int = 60) -> list[dict]:
    """
    Weighted RRF retrieval.

    Args:
        query : Retrieval query string.
        alpha : Weight for BM25. SBERT weight = (1 - alpha).
        top_k : Number of results to return.
        k     : RRF smoothing constant.

    Returns:
        List of dicts with doc_id, rrf_score (weighted), bm25_rank, sbert_rank, text.
    """
    bm25_order  = retriever._bm25_ranked(query)
    sbert_order = retriever._sbert_ranked(query)

    bm25_rank  = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_order)}
    sbert_rank = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_order)}

    rrf_scores = {}
    for doc_id in range(len(corpus)):
        rrf_scores[doc_id] = (
            alpha       * (1.0 / (k + bm25_rank[doc_id]))
            + (1-alpha) * (1.0 / (k + sbert_rank[doc_id]))
        )

    sorted_ids = sorted(rrf_scores, key=lambda d: rrf_scores[d], reverse=True)

    return [
        {
            "doc_id"    : doc_id,
            "rrf_score" : round(rrf_scores[doc_id], 6),
            "bm25_rank" : bm25_rank[doc_id],
            "sbert_rank": sbert_rank[doc_id],
            "text"      : corpus[doc_id],
        }
        for doc_id in sorted_ids[:top_k]
    ]


# --- Experiment ---
# Keyword-heavy query: BM25 should dominate (high alpha)
keyword_q  = "BM25 TF-IDF term frequency inverse document"
# Semantic query: SBERT should dominate (low alpha)
semantic_q = "how do models learn the meaning of words from context?"

for query_label, query in [("KEYWORD-HEAVY", keyword_q), ("SEMANTIC", semantic_q)]:
    print(f"\n{'='*60}")
    print(f"Query type: {query_label}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    for alpha in [0.3, 0.5, 0.7]:
        results = weighted_rrf_retrieve(query, alpha=alpha, top_k=3)
        print(f"\n  alpha={alpha} (BM25={alpha}, SBERT={1-alpha})")
        for i, r in enumerate(results, 1):
            print(f"    {i}. doc_id={r['doc_id']:02d} | {r['text'][:70]}...")


Query type: KEYWORD-HEAVY
Query: BM25 TF-IDF term frequency inverse document

  alpha=0.3 (BM25=0.3, SBERT=0.7)
    1. doc_id=11 | BM25 is a probabilistic sparse retrieval model that scores documents b...
    2. doc_id=13 | ColBERT uses late interaction between query and document token embeddi...
    3. doc_id=12 | Retrieval-Augmented Generation (RAG) grounds large language model resp...

  alpha=0.5 (BM25=0.5, SBERT=0.5)
    1. doc_id=11 | BM25 is a probabilistic sparse retrieval model that scores documents b...
    2. doc_id=13 | ColBERT uses late interaction between query and document token embeddi...
    3. doc_id=12 | Retrieval-Augmented Generation (RAG) grounds large language model resp...

  alpha=0.7 (BM25=0.7, SBERT=0.30000000000000004)
    1. doc_id=11 | BM25 is a probabilistic sparse retrieval model that scores documents b...
    2. doc_id=13 | ColBERT uses late interaction between query and document token embeddi...
    3. doc_id=12 | Retrieval-Augmented Generation (RAG) g

### Bonus 1 Observations

- **Keyword-heavy queries** (e.g., `"BM25 TF-IDF term frequency inverse document"`): Higher $\alpha$ (more BM25 weight, e.g., 0.7) surfaces the BM25 document as #1 reliably, because BM25 benefits from the exact keyword overlap.
- **Semantic queries** (e.g., `"how do models learn the meaning of words from context?"`): Lower $\alpha$ (more SBERT weight, e.g., 0.3) is better, as SBERT handles paraphrase and conceptual matching that BM25 misses.
- The symmetric default ($\alpha = 0.5$) is a safe general-purpose choice, but tuning $\alpha$ toward the expected query distribution of your application can improve precision.